# Attention Head Specialization

Classify heads by functional role: previous-token, induction,
positional, focused vs diffuse entropy profiles.

## Why This Matters

Attention head specialization reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_specialization import (
    previous_token_score, induction_score,
    positional_head_score, head_entropy_profile,
    head_specialization_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Previous Token Heads

In [ ]:
for layer in range(4):
    result = previous_token_score(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_previous_token']} prev-token heads")
    for p in result['per_head']:
        flag = ' [PREV]' if p['is_previous_token'] else ''
        print(f"    H{p['head']}: score={p['previous_token_score']:.4f}{flag}")

## Induction Heads

In [ ]:
rep_tokens = jnp.array([1, 15, 30, 1, 15, 30, 45])
for layer in range(4):
    result = induction_score(model, rep_tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_induction']} induction heads")
    for p in result['per_head']:
        flag = ' [IND]' if p['is_induction'] else ''
        print(f"    H{p['head']}: score={p['induction_score']:.4f}{flag}")

## Entropy Profiles

In [ ]:
for layer in range(4):
    result = head_entropy_profile(model, tokens, layer=layer)
    for p in result['per_head']:
        bar = '█' * int(p['normalized_entropy'] * 20)
        print(f"  L{layer}H{p['head']}: {p['classification']:7s} H={p['entropy']:.4f} {bar}")

## Summary

In [ ]:
result = head_specialization_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: prev={p['n_previous_token']}, "
          f"ind={p['n_induction']}, pos={p['n_positional']}")